# 1. 모델 학습 · 예측 저장

`data/` 3종을 읽어 **walk-forward OOS**(과거→미래, test=미래 구간)로 모델을 학습하고,
**모델별 예측**을 `result/predictions/<model>.csv` 에 저장합니다.

- DeepONet(MLP): 직접 + 하이브리드
- PI-DeepONet(MLP): 하이브리드
- DeepONet-Curve: 직접 + 하이브리드
- 벤치마크(직접): GBM/Ridge/LightGBM/CatBoost/XGBoost

하이브리드 예측 csv에는 stage별 R² 계산용 컬럼(`mc_pred`, `resid_pred` 등)이 함께 저장됩니다.

In [1]:
from util import utils, file_manager as fm
from module import data
from model import deeponet, pi_deeponet, deeponet_curve, benchmark

cfg = utils.load_config()
utils.set_seed(cfg["seed"])
D = data.load(cfg)
print(f"loaded | rows {D.n} | folds {len(D.WF)} | device {D.DEV}")

loaded | rows 23151 | folds 4 | device cuda


In [2]:
preds = {}
preds.update(deeponet.run(D, cfg))
preds.update(pi_deeponet.run(D, cfg))
preds.update(deeponet_curve.run(D, cfg))
preds.update(benchmark.run(D, cfg))

for name, df in preds.items():
    path = fm.prediction(name)
    df.to_csv(path, index=False)
    print(f"saved {name:24s} rows {len(df):5d} -> {path.relative_to(fm.ROOT)}")
print("\nDONE")

saved deeponet_direct          rows  9261 -> result\predictions\deeponet_direct.csv
saved deeponet_hybrid          rows  9261 -> result\predictions\deeponet_hybrid.csv
saved pi_deeponet_hybrid       rows  9261 -> result\predictions\pi_deeponet_hybrid.csv
saved deeponet_curve_direct    rows  9261 -> result\predictions\deeponet_curve_direct.csv
saved deeponet_curve_hybrid    rows  9261 -> result\predictions\deeponet_curve_hybrid.csv
saved bench_gbm                rows  9261 -> result\predictions\bench_gbm.csv
saved bench_ridge              rows  9261 -> result\predictions\bench_ridge.csv
saved bench_lgbm               rows  9261 -> result\predictions\bench_lgbm.csv
saved bench_catboost           rows  9261 -> result\predictions\bench_catboost.csv
saved bench_xgboost            rows  9261 -> result\predictions\bench_xgboost.csv

DONE
